# Phase 47 — Is the ESM2-650M CryptoBench Result Seed-Stable?

Phase 46 (3 seeds, ESM2-650M) found AUC 0.882 -- now exceeding
CryptoBench's own published SOTA (0.87), and closing much of the AUPRC/MCC
gap the 35M-ESM2 version (Phase43) had. Phase44 already confirmed the 35M
result was seed-stable (improved, not shrank, at 8 seeds); this notebook
applies the identical stress test to the 650M result. Trains 5 more seeds
(8 total) on Phase 46's cached training set, reports the per-seed spread
for every metric, and the 8-seed ensemble number.

## Cell 1 — Setup, load Phase 45's ESM2-650M CryptoBench blob

In [ ]:
!pip install -q torch-geometric==2.6.1

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              matthews_corrcoef, f1_score, accuracy_score)

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase45_cryptobench_graphs_esm650m.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"]); POS_WEIGHT = float(blob["pos_weight"])
splits = {s: [d for d in dataset if d.split == s] for s in ("train","val","test")}

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
print(f"device {DEVICE} | node dim {dataset[0].x.size(1)} | pos_weight {POS_WEIGHT:.2f} | " +
      " | ".join(f"{k} {len(v)}" for k,v in splits.items()))

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)

## Cell 2 — Model (identical to every prior localization phase)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 3 — Training machinery (identical to Phase 14/20/43)

In [ ]:
def run_epoch(model, loader, opt=None, lam=0.0):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    huber = nn.SmoothL1Loss(beta=4.0)                    # ~matches the 4A DCC threshold
    ps, ys, cd = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            logits, pred_c = model(b)
            loss = crit(logits, b.y)
            if lam > 0 and pred_c is not None:
                true_c = b.true_center.view(-1, 3)
                loss = loss + lam * huber(pred_c, true_c)
                cd.append((pred_c - true_c).norm(dim=1).detach().cpu().numpy())
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    out = {"pr": average_precision_score(y, p), "roc": roc_auc_score(y, p)}
    if cd:
        d_ = np.concatenate(cd)
        out["dcc"] = float((d_ < 4.0).mean()); out["mean_center_err"] = float(d_.mean())
    return out


def fit(use_loc_head, lam, seed, epochs=40, patience=8, batch=16):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=use_loc_head).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(splits["train"], batch_size=batch, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=batch)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        run_epoch(model, tl, opt, lam)
        v = run_epoch(model, vl, lam=lam)
        sch.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], {k:t.clone() for k,t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    return model, best

## Cell 4 — Load Phase 46's 3 checkpoints, train 5 MORE seeds (8 total)

In [ ]:
ensemble = {}
for seed in (0,1,2):
    cands = []
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): cands.extend(r.rglob(f"cryptobench_esm650m_deploy_seed{seed}.pt"))
    assert cands, f"cryptobench_esm650m_deploy_seed{seed}.pt not found -- attach Phase 46's kernel output"
    state = torch.load(max(cands, key=lambda p: p.stat().st_size), map_location=DEVICE)
    m = LocalizingEGNN(N_PRE, TOPO_DIM).to(DEVICE); m.load_state_dict(state); m.eval()
    ensemble[seed] = m
print(f"loaded {len(ensemble)} checkpoints from Phase 46 (no retraining)")

BEST_LAMBDA = 0.1
t0 = time.time()
for seed in (3,4,5,6,7):
    m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=seed)
    v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
    ensemble[seed] = m
    torch.save(m.state_dict(), WORK/f"cache/cryptobench_esm650m_deploy_seed{seed}.pt")
    val_dcc = v.get("dcc", float("nan"))
    print(f"  seed {seed}: val PR {val_pr:.4f}  val DCC {val_dcc:.4f}  [{(time.time()-t0)/60:.1f}m]")
print(f"\n{len(ensemble)} total seeds available (0-7)")

## Cell 5 — Per-seed and ensemble predictions on val/test

In [ ]:
@torch.no_grad()
def predict_seed(model, loader):
    """Returns pooled (probs, y, per-chain-DCC, per-chain-predicted-center,
    per-chain-true-center). Centers are kept raw (not yet turned into a
    distance) so an ensemble can average CENTERS across seeds before
    computing DCC -- averaging per-seed DISTANCES instead would be a
    different, wrong statistic (see Phase23/26's "ensemble-of-predictions
    vs mean-of-individual-scores are not the same" lesson)."""
    all_probs, all_y, all_dcc, all_centers, all_true_c = [], [], [], [], []
    for b in loader:
        b = b.to(DEVICE)
        lg, pc = model(b)
        p = torch.sigmoid(lg).cpu().numpy()
        all_probs.append(p); all_y.append(b.y.cpu().numpy())
        true_c = b.true_center.view(-1,3).cpu().numpy()
        pc_np = pc.cpu().numpy()
        d_ = np.linalg.norm(pc_np - true_c, axis=1)
        all_dcc.extend(d_.tolist())
        all_centers.append(pc_np); all_true_c.append(true_c)
    return (np.concatenate(all_probs), np.concatenate(all_y), np.array(all_dcc),
            np.concatenate(all_centers), np.concatenate(all_true_c))

val_loader = DataLoader(splits["val"], batch_size=16)
test_loader = DataLoader(splits["test"], batch_size=16)

val_preds, test_preds = {}, {}
for seed, m in ensemble.items():
    val_preds[seed] = predict_seed(m, val_loader)
    test_preds[seed] = predict_seed(m, test_loader)
print(f"predicted for {len(ensemble)} seeds x (val + test)")

## Cell 6 — Per-seed metrics (the noise floor)

In [ ]:
def metrics_at(p, y, dcc, thresh):
    pred = (p >= thresh).astype(int)
    return {
        "auc": roc_auc_score(y, p), "auprc": average_precision_score(y, p),
        "mcc": matthews_corrcoef(y, pred), "f1": f1_score(y, pred),
        "dcc_rate_4A": float((dcc < 4.0).mean()), "mean_error_A": float(dcc.mean()),
    }

# per-seed threshold tuned on that seed's own val predictions (matches Phase43's
# val-only discipline, just repeated per seed instead of once for the ensemble)
per_seed_metrics = {}
for seed in range(8):
    vp, vy, *_ = val_preds[seed]
    best_t, best_mcc = 0.5, -1.0
    for t in np.arange(0.05, 0.96, 0.01):
        mcc = matthews_corrcoef(vy, (vp >= t).astype(int))
        if mcc > best_mcc: best_mcc, best_t = mcc, t
    tp, ty, td, *_ = test_preds[seed]
    per_seed_metrics[seed] = {**metrics_at(tp, ty, td, best_t), "val_thresh": float(best_t)}

print(f"{'seed':<6}{'AUC':>8}{'AUPRC':>8}{'MCC':>8}{'F1':>8}{'DCC@4A':>8}{'thresh':>8}")
print("-"*54)
for seed in range(8):
    m_ = per_seed_metrics[seed]
    print(f"{seed:<6}{m_['auc']:>8.4f}{m_['auprc']:>8.4f}{m_['mcc']:>8.4f}{m_['f1']:>8.4f}"
          f"{m_['dcc_rate_4A']:>8.4f}{m_['val_thresh']:>8.2f}")

for key in ("auc","auprc","mcc","f1","dcc_rate_4A"):
    vals = np.array([per_seed_metrics[s][key] for s in range(8)])
    print(f"{key:<12} mean {vals.mean():.4f}  std {vals.std():.4f}  "
          f"range [{vals.min():.4f}, {vals.max():.4f}]")

## Cell 7 — 8-seed ensemble metrics (pooled predictions averaged across seeds)

In [ ]:
val_p_ens = np.mean([val_preds[s][0] for s in range(8)], axis=0)
val_y_ens = val_preds[0][1]
test_p_ens = np.mean([test_preds[s][0] for s in range(8)], axis=0)
test_y_ens = test_preds[0][1]
# ensemble DCC = distance of the AVERAGED CENTER, not the average of
# per-seed distances (a different, wrong statistic -- see predict_seed's
# docstring)
test_centers_ens = np.mean([test_preds[s][3] for s in range(8)], axis=0)
test_true_c = test_preds[0][4]
test_dcc_ens = np.linalg.norm(test_centers_ens - test_true_c, axis=1)

best_t_ens, best_mcc_ens = 0.5, -1.0
for t in np.arange(0.05, 0.96, 0.01):
    mcc = matthews_corrcoef(val_y_ens, (val_p_ens >= t).astype(int))
    if mcc > best_mcc_ens: best_mcc_ens, best_t_ens = mcc, t

ens_metrics = metrics_at(test_p_ens, test_y_ens, test_dcc_ens, best_t_ens)
print(f"8-seed ensemble (val-tuned threshold {best_t_ens:.2f}):")
for k, v in ens_metrics.items():
    print(f"  {k}: {v:.4f}")

print(f"\nPhase 46's 3-seed number for direct comparison: AUC 0.8822, AUPRC 0.3577, "
      f"MCC 0.3761, F1 0.4013, DCC@4A 0.2033")
print(f"3-seed -> 8-seed AUC delta: {ens_metrics['auc']-0.8822:+.4f}")
print(f"3-seed -> 8-seed DCC@4A delta: {ens_metrics['dcc_rate_4A']-0.2033:+.4f}")

json.dump({"per_seed": per_seed_metrics, "ensemble_8seed": ens_metrics,
           "phase46_3seed": {"auc":0.8822,"auprc":0.3577,"mcc":0.3761,"f1":0.4013,"dcc_rate_4A":0.2033}},
          open(WORK/"phase47_summary.json", "w"), indent=2)
print("\nsaved -> phase47_summary.json")

---

## Reading this

- **AUC/AUPRC/MCC hold or improve with more seeds, spread is tight** --
  Phase 46's result is real and stable; safe to cite as the headline
  CryptoBench finding.
- **Wide per-seed spread, ensemble number close to Phase46's 3-seed** --
  the model works but individual runs are noisy; report the ensemble
  number with the spread stated, not a single run.
- **8-seed number drops meaningfully below Phase46's 3-seed** -- the
  original 3-seed pass got lucky; the honest number is lower and should
  replace it in any writeup.